| Label | Definition | Impact on Speed / Flow |
| :--- | :--- | :--- |
| **no-traffic** | Road is empty or near-empty. | Free-flow at max speed. |
| **low-traffic** | Fewer vehicles are present; plenty of space | No impact on speed. |
| **medium-traffic** | More vehicles are present; reduced gaps. | **Normal speed** maintained. |
| **heavy-traffic** | Road is crowded; high density. | Forced to **slow down**. |
| **signal-loss** | Image is black or showing "No Service". | Data is unusable. |

In [ ]:
# 1. 安裝必要的套件
!pip install -q transformers datasets timm huggingface_hub evaluate pandas

In [ ]:
import os
import shutil
import time
import torch
import requests
import PIL.Image
from tqdm import tqdm
from google.colab import widgets
from IPython.display import display, clear_output
from transformers import pipeline, AutoModelForImageClassification, AutoImageProcessor, TrainingArguments, Trainer
from datasets import Dataset, Features, ClassLabel, Image as DatasetImage
import numpy as np
import pandas as pd # Ensure pandas is imported for CSV operations
from google.colab import drive # Import drive for Google Drive integration

# 設定
DATASET_URL = "https://github.com/szeu/hk-traffic-dataset/archive/refs/heads/main.zip"
IMAGE_DIR = "traffic_images"
LABELS = ['no-traffic', 'low-traffic', 'medium-traffic', 'heavy-traffic', 'signal-loss']
ITERATION_SIZE = 200 # 每次標注 200 張影像後進行一次訓練
SYNC_SAVE_INTERVAL = 20 # 每次標注 20 張影像後儲存並同步一次 CSV

# 2. 下載並解壓縮數據集
if not os.path.exists(IMAGE_DIR):
    print("下載並解壓縮中...")
    !wget -q {DATASET_URL} -O main.zip
    !unzip -q main.zip

    # 確保目標目錄存在
    os.makedirs(IMAGE_DIR, exist_ok=True)

    UNZIP_FOLDER = "hk-traffic-dataset-main"
    SEARCH_PATH = os.path.join(UNZIP_FOLDER, "traffic_snapshots")

    if os.path.exists(SEARCH_PATH):
        print(f"正在快速移動影像...")
        # 直接獲取所有檔案路徑並搬移
        files = os.listdir(SEARCH_PATH)
        for f in files:
            # 只搬圖片，避免搬到隱藏檔
            if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                shutil.move(os.path.join(SEARCH_PATH, f), os.path.join(IMAGE_DIR, f))

        # 清理垃圾
        shutil.rmtree(UNZIP_FOLDER)
        if os.path.exists("main.zip"): os.remove("main.zip")

        print(f"✅ 成功提取 {len(os.listdir(IMAGE_DIR))} 張影像至 {IMAGE_DIR}")
    else:
        print(f"❌ 找不到來源路徑: {SEARCH_PATH}")
        print("當前路徑下的資料夾有:", [d for d in os.listdir('.') if os.path.isdir(d)])

all_image_paths = [os.path.join(IMAGE_DIR, f) for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.png'))]
all_image_paths.sort()
print(f"找到 {len(all_image_paths)} 張影像。")

# 3. 載入初步分類模型 (prithivMLmods/Traffic-Density-Classification)
print("正在載入初步分類模型...")
pre_model = pipeline("image-classification", model="prithivMLmods/Traffic-Density-Classification", device=0)

# 4. 互動式標注函式
def label_images(image_list):
    labeled_data = []
    idx = 0 # Current index within the image_list batch
    while idx < len(image_list):
        img_path = image_list[idx]
        clear_output(wait=True)
        img = PIL.Image.open(img_path)
        display(img.resize((400, 300)))

        # 初步預測
        pred = pre_model(img)[0]['label'].lower()
        # 映射標籤 (若名稱不完全相符則需手動修正)
        mapped_pred = pred if pred in LABELS else "未匹配"

        print(f"路徑: {img_path}")
        print(f"預測標籤: {mapped_pred}")
        print(f"標籤選項: 0:{LABELS[0]}, 1:{LABELS[1]}, 2:{LABELS[2]}, 3:{LABELS[3]}, 4:{LABELS[4]}, b:上一張影像") # Added 'b' option
        time.sleep(.5)
        user_input = input("輸入數字選擇標籤，或直接按回車(Enter)接受預測標籤: \n").strip().lower()

        if user_input == 'b':
            if idx > 0:
                idx -= 1
                # Remove the last labeled item if we are going back to re-label it
                if labeled_data and os.path.join(IMAGE_DIR, labeled_data[-1]['image']) == image_list[idx]:
                    labeled_data.pop()
                continue # Go to the previous image
            else:
                print("已經是第一張影像了，無法再返回。")
                time.sleep(1)
                continue # Stay on the first image

        if user_input == "" and mapped_pred in LABELS:
            final_label = mapped_pred
        elif user_input in [str(i) for i in range(len(LABELS))]: # Dynamically check valid label numbers
            final_label = LABELS[int(user_input)]
        else:
            final_label = "low-traffic" # 預設或錯誤處理

        # Store only the basename (filename) in labeled_data
        labeled_data.append({"image": os.path.basename(img_path), "label": final_label})
        idx += 1 # Move to the next image
    return labeled_data

# 5. 微調 DeiT 模型函式
def train_deit(labeled_results, iteration_num):
    print(f"\n--- 開始第 {iteration_num} 輪 DeiT 微調 (數據量: {len(labeled_results)}) ---")

    # 1. 準備 Processor
    model_id = "facebook/deit-base-distilled-patch16-224"
    processor = AutoImageProcessor.from_pretrained(model_id)

    # 2. 手動預處理所有影像 (避免 JpegImageFile 錯誤)
    pixel_values = []
    labels = []

    print("正在預處理影像...")
    # Use tqdm for progress bar
    for item in tqdm(labeled_results, desc="預處理影像"):
        img_filename = item['image']
        label_str = item['label']

        # Reconstruct the full path for opening the image
        img_path_full = os.path.join(IMAGE_DIR, img_filename)
        
        # 讀取並轉換
        img = PIL.Image.open(img_path_full).convert("RGB")
        inputs = processor(img, return_tensors="pt")

        # 關鍵修改：轉為 Numpy 並存儲
        pixel_values.append(inputs.pixel_values.squeeze(0).numpy())
        labels.append(LABELS.index(label_str))

    # 3. 建立數據集 (使用 Numpy Array)
    full_ds = Dataset.from_dict({
        "pixel_values": np.array(pixel_values, dtype=np.float32),
        "label": labels  # 修正處：從 labels 改為 label
    })

    # 4. 分割訓練與測試集
    ds_split = full_ds.train_test_split(test_size=0.2, seed=42)
    train_ds = ds_split['train']
    test_ds = ds_split['test']

    # 5. 載入模型
    model = AutoModelForImageClassification.from_pretrained(
        model_id,
        num_labels=len(LABELS),
        id2label={i: l for i, l in enumerate(LABELS)},
        label2id={l: i for i, l in enumerate(LABELS)},
        ignore_mismatched_sizes=True
    ).to("cuda")

    # 6. 設定訓練參數
    training_args = TrainingArguments(
        output_dir=f"./deit-traffic-it{iteration_num}",
        remove_unused_columns=False,
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=5e-5,
        per_device_train_batch_size=8,
        num_train_epochs=10,
        logging_steps=10,
        report_to="none",
        # 移除 label_names 參數，改用下方的自定義 Trainer 處理
    )

     # 7. 建立自定義 Trainer 手動計算 Loss
    class DeiTTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            # 提取標籤並從 inputs 中移除
            if "labels" in inputs:
                labels = inputs.pop("labels")
            elif "label" in inputs:
                labels = inputs.pop("label")
            else:
                labels = None

            # 取得模型輸出
            outputs = model(**inputs)

            # DeiT Distilled 輸出包含 logits (通常是 cls_logits 和 distillation_logits 的平均或組合)
            logits = outputs.logits

            # 手動計算 Cross Entropy Loss
            loss_fct = torch.nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))

            return (loss, outputs) if return_outputs else loss

    def compute_metrics(eval_pred):
        # 這裡 eval_pred.predictions 會包含所有 logits
        # 對於 DeiT Distilled，通常 index 0 是最終可以用來分類的 logits
        predictions, labels = eval_pred
        if isinstance(predictions, (list, tuple)):
          predictions = predictions[0]

        preds = np.argmax(predictions, axis=-1)
        return {"accuracy": (preds == labels).astype(np.float32).mean().item()}

    # 使用自定義的 DeiTTrainer 啟動
    trainer = DeiTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    accuracy = metrics.get('eval_accuracy') or metrics.get('accuracy') or 0.0

    print(f"\n✅ 第 {iteration_num} 輪結果: Accuracy = {accuracy:.4f}")
    print(f"完整指標: {metrics}")

    return model

# 6. 主執行迴圈 (疊代標注與訓練)
drive.mount('/content/drive')

total_labeled = []
DRIVE_CSV_PATH = "/content/drive/MyDrive/labeled_traffic_results.csv"

# Load existing data from Drive if available
if os.path.exists(DRIVE_CSV_PATH):
    print(f"從 Google Drive 載入現有標注結果: {DRIVE_CSV_PATH}")
    try:
        df_existing = pd.read_csv(DRIVE_CSV_PATH)
        # Ensure all loaded 'image' paths are basenames
        total_labeled.extend([{"image": os.path.basename(row['image']), "label": row['label']} for index, row in df_existing.iterrows()])
        print(f"已載入 {len(total_labeled)} 筆現有標注。")
    except Exception as e:
        print(f"載入 Google Drive CSV 失敗: {e}. 將從頭開始標注。")
        total_labeled = [] # Reset if loading fails

# Filter out already labeled images from `all_image_paths`
# processed_image_paths now contains only filenames
processed_image_paths = {item['image'] for item in total_labeled}
remaining_image_paths = [path for path in all_image_paths if os.path.basename(path) not in processed_image_paths]
print(f"找到 {len(remaining_image_paths)} 張待標注影像。")

# Keep track of the last training point for total_labeled
last_training_point = (len(total_labeled) // ITERATION_SIZE) * ITERATION_SIZE

for i in range(0, len(remaining_image_paths), SYNC_SAVE_INTERVAL):
    batch = remaining_image_paths[i : i + SYNC_SAVE_INTERVAL]

    print(f"\n現在開始標注 {len(batch)} 張影像 (目前累計標注 {len(total_labeled)} 張)...")
    new_labeled_batch = label_images(batch)
    total_labeled.extend(new_labeled_batch)

    # Save locally
    print(f"儲存 {len(total_labeled)} 筆標注結果到本地 CSV...")
    df_current_labels = pd.DataFrame(total_labeled)
    df_current_labels.to_csv("labeled_traffic_results.csv", index=False)

    # Synchronize to Google Drive
    print(f"嘗試同步 {len(total_labeled)} 筆標注結果到 Google Drive...")
    try:
        df_current_labels.to_csv(DRIVE_CSV_PATH, index=False)
        print("同步到 Google Drive 成功。")
    except Exception as e:
        print(f"同步到 Google Drive 失敗: {e}. 繼續執行。")

    # Check for model training condition
    if len(total_labeled) >= last_training_point + ITERATION_SIZE:
        training_iteration_num = len(total_labeled) // ITERATION_SIZE
        print(f"\n--- 已累計 {len(total_labeled)} 張影像，將進行第 {training_iteration_num} 輪模型微調。---")
        current_model = train_deit(total_labeled, training_iteration_num)
        last_training_point = training_iteration_num * ITERATION_SIZE
        input("按下 Enter 鍵開始下一階段標注...\n")
    elif i + SYNC_SAVE_INTERVAL >= len(remaining_image_paths): # End of all images
        if len(total_labeled) > last_training_point: # Train if there's any new data since last training
            training_iteration_num = (len(total_labeled) // ITERATION_SIZE) + (1 if len(total_labeled) % ITERATION_SIZE != 0 else 0)
            print(f"\n--- 所有影像已標注完畢 ({len(total_labeled)} 張)。將進行第 {training_iteration_num} 輪模型微調。---")
            current_model = train_deit(total_labeled, training_iteration_num)
        print("所有標注與訓練已完成。結果儲存在 labeled_traffic_results.csv")
    else:
        input("按下 Enter 鍵開始下一批標注...\n")

print("所有標注與訓練已完成。結果儲存在 labeled_traffic_results.csv")


In [ ]:
import pandas as pd
import os

# --- Settings ---
csv_filename = 'labeled_traffic_results.csv'        # Update with your CSV name
img_folder_name = 'traffic_snapshots'  # The folder containing images
output_dir = 'table_reports'
chunk_size = 50

def generate_html_reports(csv_path, img_folder, size, out_dir):
    if not os.path.exists(csv_path):
        print(f"Error: {csv_path} not found.")
        return
    
    df = pd.read_csv(csv_path)

    df['prefix'] = df.iloc[:, 0].apply(lambda x: str(x).split('_')[0])
    df.sort_values(by=['prefix', df.columns[1]], inplace=True)
    df.drop(columns=['prefix'], inplace=True)

    if not os.path.exists(out_dir):
        os.makedirs(out_dir)

    num_chunks = (len(df) + size - 1) // size

    for i in range(num_chunks):
        chunk = df.iloc[i * size : (i + 1) * size]
        
        # HTML with CSS for a clean look
        html_content = f"""
        <html>
        <head>
            <style>
                body {{ font-family: Arial, sans-serif; margin: 20px; }}
                table {{ border-collapse: collapse; width: 100%; max-width: 800px; }}
                th, td {{ border: 1px solid #ddd; padding: 12px; text-align: left; }}
                th {{ background-color: #f4f4f4; }}
                img {{ max-width: 150px; height: auto; display: block; border-radius: 4px; }}
                .filename {{ font-size: 0.85em; color: #666; margin-top: 5px; }}
            </style>
        </head>
        <body>
            <h2>Traffic Data Report - Part {i+1}</h2>
            <table>
                <tr>
                    <th>Image Preview</th>
                    <th>Label</th>
                </tr>
        """

        for _, row in chunk.iterrows():
            # Adjust these column names to match your CSV headers exactly
            img_name = str(row.iloc[0]) 
            label = str(row.iloc[1])
            
            # Using your specific relative path requirement
            img_src = f"../{img_folder}/{img_name}"
            
            html_content += f"""
                <tr>
                    <td>
                        <img src="{img_src}">
                        <div class="filename">{img_name}</div>
                    </td>
                    <td><strong>{label}</strong></td>
                </tr>
            """

        html_content += "</table></body></html>"

        # Save the file
        output_file = os.path.join(out_dir, f"report_part_{i+1}.html")
        with open(output_file, 'w') as f:
            f.write(html_content)
        
        print(f"✅ Created: {output_file}")

# Run
generate_html_reports(csv_filename, img_folder_name, chunk_size, output_dir)
